# W7 · Day 2 — Qdrant Tour: Indices, Metrics, Mini-RAG

**~90 minutes · in-class demo · Jupyter notebook · Track A**

Day 1 you connected to Qdrant and made your first queries. Day 2 goes
deeper: what does HNSW actually do, why does the similarity metric matter,
and how do you assemble a full mini-RAG on top of Qdrant.

Same animals corpus as Day 1 (imported from `wk07_pipeline.py`). We build
on top of it rather than rebuilding.

**Cost per full run:** ~$0.02 (we build a few collections; a bit more OpenAI).

**Notebook flow:**
- Cell 1: Setup + import Day 1's helpers
- Cell 2: Confirm Qdrant reachable + Day 1 collection still exists
- Cells 3-4: The problem indices solve (linear scan timing)
- Cells 5-7: HNSW — Qdrant's default index; how tuning affects recall/speed
- Cells 8-9: Qdrant's quantization mode (IVF's philosophical cousin)
- Cells 10-12: Similarity metrics — cosine vs dot vs L2 on the same corpus
- Cell 13: The silent-bug pattern — non-normalised vectors + wrong metric
- Cells 14-16: Build a mini-RAG on top of Qdrant (5 test questions)
- Cell 17: Wrap + hand-off to Track B

---

## Cell 1 — Setup + import Day 1's pipeline

The helper module `wk07_pipeline.py` contains the same corpus, embedding
functions, and similarity metrics from Day 1. We import them so Day 2
doesn't re-embed everything from scratch.

Open `wk07_pipeline.py` alongside if you want to see what you're importing.

In [16]:
import os
import time
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
import importlib

# Load key-value pairs from the .env file into os.environ
load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook"
assert os.environ.get("QDRANT_URL"),     "Set QDRANT_URL — get free-tier at cloud.qdrant.io"
assert os.environ.get("QDRANT_API_KEY"), "Set QDRANT_API_KEY — from your Qdrant Cloud cluster"

# Force reload the pipeline
import wk07_pipeline

print("Pipeline file:")
print(wk07_pipeline.__file__)

importlib.reload(wk07_pipeline)


from wk07_pipeline import (
    CORPUS_ANIMALS,
    TEST_QUESTIONS,
    embed_batch,
    embed_one,
    cosine, dot, l2,
    get_qdrant_client,
    EMBED_SMALL,
)

openai_client = OpenAI()

print(f"Loaded {len(CORPUS_ANIMALS)} animals from Day 1's helper.")
print(f"Loaded {len(TEST_QUESTIONS)} test questions.")
print("Metrics available: cosine, dot, l2")
print("Pipeline functions: embed_batch, embed_one, get_qdrant_client")

Pipeline file:
/voc/work/IITM_Agentic_AI_Code/Knowledge_Hands_on/practice_examples/week07/wk07_pipeline.py
Loaded 16 documents
data/goldenSet.json
Loaded 77 golden-set questions
ID: 1
Question: Where is the company located?
--------------------------------------------------------------------------------
ID: 2
Question: What is the difference between Open positions and closed positions?
--------------------------------------------------------------------------------
ID: 3
Question: On what basis the clients are categorized as Partially Approved?
--------------------------------------------------------------------------------
ID: 4
Question: Is my telephone call recorded?
--------------------------------------------------------------------------------
ID: 5
Question: What are the limitations on withdrawals?
--------------------------------------------------------------------------------
ID: 6
Question: What is a leverage and how can i change it?
------------------------------------------

---

## Cell 2 — Reconnect to Qdrant + embed the corpus

Same cluster as Day 1. We'll build several collections today, each
demonstrating a different configuration.

In [23]:
qdrant = get_qdrant_client()

COLLECTION_NAME = "wk07_day1_mycorpus_hfm"

# Sanity: what collections already exist?
existing = [c.name for c in qdrant.get_collections().collections]
print(f"Existing collections: {existing}\n")

#we don't need to re-embed data already embedded and stored in Qdrant
info = qdrant.get_collection(
    collection_name=COLLECTION_NAME
)


print(f"Using existing collection: {COLLECTION_NAME}")
print(f"Vector dimension: {info.config.params.vectors.size}")
print(f"Distance metric: {info.config.params.vectors.distance}")
print(f"Points in collection: {info.points_count}")

# # Embed all 10 animals fresh (Day 1's variables aren't in scope here)
# print("Embedding animals with text-embedding-3-small...")
# texts = [d["text"] for d in CORPUS_ANIMALS]
# vectors = embed_batch(texts, model=EMBED_SMALL)

# # Attach to records
# for doc, vec in zip(CORPUS_ANIMALS, vectors):
#     doc["vector"] = vec

# print(
#     f"Embedded {len(vectors)} corpus documents "
#     f"({len(vectors[0])} dims each)."
# )

Existing collections: ['wk07_day1_animals', 'wk07_day1_mycorpus_hfm']

Using existing collection: wk07_day1_mycorpus_hfm
Vector dimension: 1536
Distance metric: Cosine
Points in collection: 1832


---

## Cell 3 — The problem indices solve

Why do vector databases need special indices? Because **linear scan doesn't
scale**. Let's see what a linear scan actually looks like.

**Linear scan** = compute cosine similarity between the query and EVERY
vector in the collection, then sort. That's O(n × d) where n is corpus size
and d is dimension.

At 10 vectors it's instant. At 10 million, it's minutes.

In [28]:
# ------------------------------------------------------------
# Load all chunk vectors from Qdrant for local linear scan
# ------------------------------------------------------------

linear_corpus = []

offset = None

while True:
    points, next_offset = qdrant.scroll(
        collection_name="wk07_day1_mycorpus_hfm",
        limit=100,
        offset=offset,
        with_payload=True,
        with_vectors=True
    )

    for point in points:
        linear_corpus.append({
            "id": point.payload.get("chunk_id", str(point.id)),
            "source": point.payload.get("source", ""),
            "text": point.payload.get("text", ""),
            "vector": point.vector
        })

    if next_offset is None:
        break

    offset = next_offset


print(f"Loaded {len(linear_corpus)} chunks from Qdrant for linear scan.")
print(f"Vector dimension: {len(linear_corpus[0]['vector'])}")

def linear_scan(query_vec, corpus, k=3):
    """Naive linear scan — compute similarity vs every doc, sort, take top-K."""
    scored = [(cosine(query_vec, doc["vector"]), doc) for doc in corpus]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored[:k]

query = TEST_QUESTIONS[0]
q_vec = embed_one(query)

# Time the linear scan on our 10-doc corpus
t0 = time.time()
hits = linear_scan(q_vec, linear_corpus, k=3)
dt = time.time() - t0

print(f"Q: {query!r}")

print(
    f"Linear scan over {len(linear_corpus)} chunks "
    f"took {dt * 1000:.2f} ms\n"
)


for rank, (score, doc) in enumerate(hits, 1):

    print(
        f"[{rank}] "
        f"score={score:.3f}  "
        f"chunk={doc['id']}  "
        f"source={doc['source']}"
    )

Loaded 1832 chunks from Qdrant for linear scan.
Vector dimension: 1536
Q: 'Where is the company located?'
Linear scan over 1832 chunks took 168.37 ms

[1] score=0.458  chunk=Risk with CFDS 2.txt#6  source=
[2] score=0.457  chunk=General risk disclosure(1).txt#5  source=
[3] score=0.451  chunk=Risk with CFDS.txt#6  source=


In [30]:
# # Now simulate a bigger corpus to see how linear scan scales.
# # We fake it by copying our 10 vectors many times.

# ------------------------------------------------------------
# Test how linear scan scales using our REAL HFM corpus
# ------------------------------------------------------------

print(
    f"  {'Corpus size':>12s}  "
    f"{'Linear scan time':>18s}  "
    f"{'ops (approx)':>15s}"
)

print(
    f"  {'-----------':>12s}  "
    f"{'----------------':>18s}  "
    f"{'-----------':>15s}"
)


# Test different sizes using the actual corpus
test_sizes = [10, 100, 1_000, len(linear_corpus)]

for size in test_sizes:

    # Use actual HFM chunks
    corpus_subset = linear_corpus[:size]

    t0 = time.time()

    scored = [
        (cosine(q_vec, d["vector"]), d)
        for d in corpus_subset
    ]

    scored.sort(
        key=lambda x: x[0],
        reverse=True
    )

    _ = scored[:3]

    dt = time.time() - t0

    # Each text-embedding-3-small vector has 1536 dimensions
    ops = len(corpus_subset) * len(q_vec)

    print(
        f"  {len(corpus_subset):>12,d}  "
        f"{dt * 1000:>15.1f} ms  "
        f"{ops:>15,d}"
    )


print()

print(
    f"Linear scan tested using the actual "
    f"{len(linear_corpus):,} HFM corpus chunks."
)

print(
    "As the number of vectors increases, linear scan must compare "
    "the query against every vector."
)

print(
    "\nSolution: pre-build an index that can avoid comparing "
    "the query against every vector at query time."
)
# print(f"  {'Corpus size':>12s}  {'Linear scan time':>18s}  {'ops (approx)':>15s}")
# print(f"  {'-----------':>12s}  {'----------------':>18s}  {'-----------':>15s}")

# for size in [10, 100, 1_000, 10_000]:
#     # Build a fake corpus by tiling our 10 vectors
#     fake_corpus = [{"vector": CORPUS_ANIMALS[i % 10]["vector"],
#                     "id": f"fake_{i}"}
#                    for i in range(size)]
    
#     t0 = time.time()
#     scored = [(cosine(q_vec, d["vector"]), d) for d in fake_corpus]
#     scored.sort(key=lambda x: x[0], reverse=True)
#     _ = scored[:3]
#     dt = time.time() - t0
    
#     ops = size * 1536
#     print(f"  {size:>12,d}  {dt*1000:>15.1f} ms  {ops:>15,d}")

# print()
# print("At 10K docs, linear scan is still workable (~10s).")
# print("At 100K, it's painful. At 10M — completely impractical.")
# print("\nSolution: pre-build an *index* that skips most of the work at query time.")

   Corpus size    Linear scan time     ops (approx)
   -----------    ----------------      -----------
            10              1.6 ms           15,360
           100              8.9 ms          153,600
         1,000             88.4 ms        1,536,000
         1,832            166.4 ms        2,813,952

Linear scan tested using the actual 1,832 HFM corpus chunks.
As the number of vectors increases, linear scan must compare the query against every vector.

Solution: pre-build an index that can avoid comparing the query against every vector at query time.


---

## Cell 4 — What indices trade off

**Linear scan is exact.** It compares to every vector, so it always finds the
true top-K. But it's O(n).

**Indexed search is approximate.** It skips most vectors using clever
structure. It might miss the true top-K sometimes, but it's O(log n) or
faster.

The trade-off is **recall vs speed**:
- Perfect recall (100%) = linear scan = slow
- 99% recall = HNSW default = fast
- 95% recall = aggressive index tuning = very fast

In production, 95-99% recall is usually fine — you're already retrieving
top-K, and one occasionally-missed candidate is acceptable.

**Two main index families:**
- **HNSW** — graph-based. Qdrant's default. Fast queries.
- **IVF** — cluster-based. Fast build. Good for very large corpora.

Next cells: HNSW in action.

---

## Cell 5 — HNSW with Qdrant defaults

Create a Qdrant collection using HNSW (the default). Upsert our 10 animals.
Query and observe.

In [31]:
from qdrant_client.models import (
    Distance,
    VectorParams,
    HnswConfigDiff,
    PointStruct
)


def recreate_collection(
    name,
    size=1536,
    distance=Distance.COSINE,
    hnsw=None
):
    """Delete-then-create so cells are re-runnable."""

    try:
        qdrant.delete_collection(name)
    except Exception:
        pass

    qdrant.create_collection(
        collection_name=name,
        vectors_config=VectorParams(
            size=size,
            distance=distance
        ),
        hnsw_config=hnsw,
    )


def upsert_corpus(collection_name):
    """Push the HFM corpus chunks into a collection."""

    BATCH_SIZE = 100

    for start in range(0, len(linear_corpus), BATCH_SIZE):

        end = min(
            start + BATCH_SIZE,
            len(linear_corpus)
        )

        batch = linear_corpus[start:end]

        points = [
            PointStruct(
                id=start + idx,
                vector=doc["vector"],
                payload={
                    "chunk_id": doc["id"],
                    "source": doc["source"],
                    "text": doc["text"]
                },
            )
            for idx, doc in enumerate(batch)
        ]

        qdrant.upsert(
            collection_name=collection_name,
            points=points,
            wait=True
        )

        print(
            f"Uploaded {start + 1:4d} - {end:4d} "
            f"({end}/{len(linear_corpus)})"
        )


# ------------------------------------------------------------
# Create Day 2 HNSW collection using Qdrant defaults
# ------------------------------------------------------------

COLL_HNSW_DEFAULT = "wk07_day2_hnsw_default"

recreate_collection(
    COLL_HNSW_DEFAULT,
    size=len(linear_corpus[0]["vector"])
)


# Upload our actual HFM corpus chunks
upsert_corpus(COLL_HNSW_DEFAULT)


# ------------------------------------------------------------
# Show the default HNSW parameters Qdrant used
# ------------------------------------------------------------

info = qdrant.get_collection(
    COLL_HNSW_DEFAULT
)

hnsw = info.config.hnsw_config


print(
    f"\nCollection {COLL_HNSW_DEFAULT!r} "
    f"— HNSW default config:"
)

print(
    f"  m:                    {hnsw.m} "
    "(graph connectivity — higher = more edges per node)"
)

print(
    f"  ef_construct:         {hnsw.ef_construct} "
    "(build-time search depth — higher = better recall, slower build)"
)

print(
    f"  full_scan_threshold:  {hnsw.full_scan_threshold} "
    "(below this many points, use linear scan)"
)

print(
    f"  points:               {info.points_count}"
)


# from qdrant_client.models import Distance, VectorParams, HnswConfigDiff, PointStruct

# def recreate_collection(name, size=1536, distance=Distance.COSINE, hnsw=None):
#     """Delete-then-create so cells are re-runnable."""
#     try:
#         qdrant.delete_collection(name)
#     except Exception:
#         pass
#     qdrant.create_collection(
#         collection_name=name,
#         vectors_config=VectorParams(size=size, distance=distance),
#         hnsw_config=hnsw,
#     )

# def upsert_corpus(collection_name):
#     BATCH_SIZE = 100
#     """Push the 10 animals into a collection."""
#     points = [
#         PointStruct(
#             id=idx,
#             vector=doc["vector"],
#             payload={"animal_id": doc["id"], "category": doc["category"], "text": doc["text"]},
#         )
#         for idx, doc in enumerate(CORPUS_ANIMALS)
#     ]
#     qdrant.upsert(collection_name=collection_name, points=points)

# COLL_HNSW_DEFAULT = "wk07_day2_hnsw_default"
# recreate_collection(COLL_HNSW_DEFAULT)  # HNSW is Qdrant's default; no config = defaults
# upsert_corpus(COLL_HNSW_DEFAULT)

# # Show the default HNSW parameters Qdrant used
# info = qdrant.get_collection(COLL_HNSW_DEFAULT)
# hnsw = info.config.hnsw_config
# print(f"Collection {COLL_HNSW_DEFAULT!r} — HNSW default config:")
# print(f"  m:               {hnsw.m}          (graph connectivity — higher = more edges per node)")
# print(f"  ef_construct:    {hnsw.ef_construct}         (build-time search depth — higher = better recall, slower build)")
# print(f"  full_scan_threshold: {hnsw.full_scan_threshold}   (below this many points, use linear scan)")

Uploaded    1 -  100 (100/1832)
Uploaded  101 -  200 (200/1832)
Uploaded  201 -  300 (300/1832)
Uploaded  301 -  400 (400/1832)
Uploaded  401 -  500 (500/1832)
Uploaded  501 -  600 (600/1832)
Uploaded  601 -  700 (700/1832)
Uploaded  701 -  800 (800/1832)
Uploaded  801 -  900 (900/1832)
Uploaded  901 - 1000 (1000/1832)
Uploaded 1001 - 1100 (1100/1832)
Uploaded 1101 - 1200 (1200/1832)
Uploaded 1201 - 1300 (1300/1832)
Uploaded 1301 - 1400 (1400/1832)
Uploaded 1401 - 1500 (1500/1832)
Uploaded 1501 - 1600 (1600/1832)
Uploaded 1601 - 1700 (1700/1832)
Uploaded 1701 - 1800 (1800/1832)
Uploaded 1801 - 1832 (1832/1832)

Collection 'wk07_day2_hnsw_default' — HNSW default config:
  m:                    16 (graph connectivity — higher = more edges per node)
  ef_construct:         100 (build-time search depth — higher = better recall, slower build)
  full_scan_threshold:  10000 (below this many points, use linear scan)
  points:               1832


**Note the `full_scan_threshold`.** With only 10 animals, Qdrant will actually
still do a linear scan — HNSW only kicks in above the threshold. This is
smart: at tiny corpora, linear IS faster than index traversal.

For our demo we're seeing HNSW's CONFIGURATION, not its behaviour. To see
HNSW's behaviour you'd need 10K+ vectors. Trust the mechanism; understand
the parameters.

---

## Cell 6 — HNSW tuned for higher recall

The two knobs that matter for HNSW:
- **`m`** (default 16) — how many graph edges per node. More edges = better
  recall, more memory, slower build.
- **`ef_construct`** (default 100) — how thoroughly the index is built.
  Higher = better recall, slower build.

Build a second collection with aggressive parameters and see how the config
changes.

In [32]:
COLL_HNSW_TUNED = "wk07_day2_hnsw_tuned"
recreate_collection(
    COLL_HNSW_TUNED,
    hnsw=HnswConfigDiff(m=32, ef_construct=200),  # 2× default on both
)
upsert_corpus(COLL_HNSW_TUNED)

info = qdrant.get_collection(COLL_HNSW_TUNED)
hnsw = info.config.hnsw_config
print(f"Collection {COLL_HNSW_TUNED!r} — HNSW tuned config:")
print(f"  m:            {hnsw.m}  (was 16, now 32)")
print(f"  ef_construct: {hnsw.ef_construct}  (was 100, now 200)")
print()
print("Effect at production scale:")
print("  - Better recall (misses fewer true top-K on approximate queries)")
print("  - More memory (each node has 2× the edges to store)")
print("  - Slower build time (index construction does 2× more work)")
print("  - Query speed roughly the same (HNSW query cost is O(log n × ef))")

Uploaded    1 -  100 (100/1832)
Uploaded  101 -  200 (200/1832)
Uploaded  201 -  300 (300/1832)
Uploaded  301 -  400 (400/1832)
Uploaded  401 -  500 (500/1832)
Uploaded  501 -  600 (600/1832)
Uploaded  601 -  700 (700/1832)
Uploaded  701 -  800 (800/1832)
Uploaded  801 -  900 (900/1832)
Uploaded  901 - 1000 (1000/1832)
Uploaded 1001 - 1100 (1100/1832)
Uploaded 1101 - 1200 (1200/1832)
Uploaded 1201 - 1300 (1300/1832)
Uploaded 1301 - 1400 (1400/1832)
Uploaded 1401 - 1500 (1500/1832)
Uploaded 1501 - 1600 (1600/1832)
Uploaded 1601 - 1700 (1700/1832)
Uploaded 1701 - 1800 (1800/1832)
Uploaded 1801 - 1832 (1832/1832)
Collection 'wk07_day2_hnsw_tuned' — HNSW tuned config:
  m:            32  (was 16, now 32)
  ef_construct: 200  (was 100, now 200)

Effect at production scale:
  - Better recall (misses fewer true top-K on approximate queries)
  - More memory (each node has 2× the edges to store)
  - Slower build time (index construction does 2× more work)
  - Query speed roughly the same (HNSW 

---

## Cell 7 — Query both collections; results should match

At our tiny scale, both collections should return identical top-3 rankings
(because both fall back to linear scan below `full_scan_threshold`).

At production scale, the tuned collection would have slightly better recall
at the cost of more memory. The pattern to learn: **HNSW parameters give you
a recall-vs-cost lever.**

In [36]:
query = TEST_QUESTIONS[0]
q_vec = embed_one(query)

for coll in [COLL_HNSW_DEFAULT, COLL_HNSW_TUNED]:
    hits = qdrant.query_points(collection_name=coll, query=q_vec, limit=3).points
    print(f"  {coll}:")
    for rank, h in enumerate(hits, 1):
        p = h.payload
        print(
            f"  [{rank}] "
            f"score={h.score:.3f}  "
            f"chunk={p.get('chunk_id', '')}  "
            f"source={p.get('source', '')}"
        )
    print()

  wk07_day2_hnsw_default:
  [1] score=0.458  chunk=Risk with CFDS 2.txt#6  source=
  [2] score=0.457  chunk=General risk disclosure(1).txt#5  source=
  [3] score=0.451  chunk=Risk with CFDS.txt#6  source=

  wk07_day2_hnsw_tuned:
  [1] score=0.458  chunk=Risk with CFDS 2.txt#6  source=
  [2] score=0.457  chunk=General risk disclosure(1).txt#5  source=
  [3] score=0.451  chunk=Risk with CFDS.txt#6  source=



**Discussion moment:**
- Did both collections return the same top-3? (They should, at this scale.)
- If they differ, that would indicate HNSW's approximation kicking in.
- **The point of this exercise isn't the results — it's the parameters.** You
  now know Qdrant exposes `m` and `ef_construct` as knobs, and what each
  controls.

---

## Cell 8 — Qdrant quantization (IVF's philosophical cousin)

**IVF** (Inverted File Index) works by clustering vectors first, then only
searching within nearby clusters. Qdrant doesn't ship IVF specifically, but
it ships a related idea: **quantization** — compressing vectors to smaller
representations for faster search.

Both approaches trade some accuracy for a lot of memory + speed.

**Qdrant supports two quantization modes:**
- **Scalar quantization** — each float32 → int8 (4× memory reduction)
- **Product quantization** — vectors split into subvectors and compressed (up to 64× reduction)

We'll enable scalar quantization on a collection and observe the config.

In [38]:
from qdrant_client.models import ScalarQuantization, ScalarQuantizationConfig, ScalarType

COLL_QUANT = "wk07_day2_quantized"

try:
    qdrant.delete_collection(COLL_QUANT)
except Exception:
    pass

VECTOR_SIZE = len(linear_corpus[0]["vector"])

qdrant.create_collection(
    collection_name=COLL_QUANT,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    quantization_config=ScalarQuantization(
        scalar=ScalarQuantizationConfig(
            type=ScalarType.INT8,
            quantile=0.99,
            always_ram=True,
        )
    ),
)
upsert_corpus(COLL_QUANT)

info = qdrant.get_collection(COLL_QUANT)
print(f"Collection {COLL_QUANT!r} created with scalar quantization.")
print(f"  Storage per vector before:  1536 × 4 bytes = 6144 bytes")
print(f"  Storage per vector after:   1536 × 1 byte  = 1536 bytes  (4× smaller)")
print()
print("On our 10-animal corpus this saves ~45 KB — trivial.")
print("On 10M vectors, this saves 45 GB. That's when quantization pays off.")

# ------------------------------------------------------------
# Storage comparison
# ------------------------------------------------------------

num_vectors = len(linear_corpus)

float32_per_vector = VECTOR_SIZE * 4
int8_per_vector = VECTOR_SIZE * 1

float32_total = num_vectors * float32_per_vector
int8_total = num_vectors * int8_per_vector


print(
    f"  Storage per vector before: "
    f"{VECTOR_SIZE} × 4 bytes = "
    f"{float32_per_vector:,} bytes"
)

print(
    f"  Storage per vector after:  "
    f"{VECTOR_SIZE} × 1 byte = "
    f"{int8_per_vector:,} bytes "
    f"(4× smaller)"
)

print()

print(f"For our actual {num_vectors:,}-chunk corpus:")

print(
    f"  Before quantization: "
    f"~{float32_total / (1024 * 1024):.2f} MB"
)

print(
    f"  After quantization:  "
    f"~{int8_total / (1024 * 1024):.2f} MB"
)

print(
    f"  Approx. vector-data saving: "
    f"~{(float32_total - int8_total) / (1024 * 1024):.2f} MB"
)

print()

print(
    "Quantization becomes increasingly valuable "
    "as the number of stored vectors grows."
)

Uploaded    1 -  100 (100/1832)
Uploaded  101 -  200 (200/1832)
Uploaded  201 -  300 (300/1832)
Uploaded  301 -  400 (400/1832)
Uploaded  401 -  500 (500/1832)
Uploaded  501 -  600 (600/1832)
Uploaded  601 -  700 (700/1832)
Uploaded  701 -  800 (800/1832)
Uploaded  801 -  900 (900/1832)
Uploaded  901 - 1000 (1000/1832)
Uploaded 1001 - 1100 (1100/1832)
Uploaded 1101 - 1200 (1200/1832)
Uploaded 1201 - 1300 (1300/1832)
Uploaded 1301 - 1400 (1400/1832)
Uploaded 1401 - 1500 (1500/1832)
Uploaded 1501 - 1600 (1600/1832)
Uploaded 1601 - 1700 (1700/1832)
Uploaded 1701 - 1800 (1800/1832)
Uploaded 1801 - 1832 (1832/1832)
Collection 'wk07_day2_quantized' created with scalar quantization.
  Storage per vector before:  1536 × 4 bytes = 6144 bytes
  Storage per vector after:   1536 × 1 byte  = 1536 bytes  (4× smaller)

On our 10-animal corpus this saves ~45 KB — trivial.
On 10M vectors, this saves 45 GB. That's when quantization pays off.
  Storage per vector before: 1536 × 4 bytes = 6,144 bytes
  St

---

## Cell 9 — Query the quantized collection

Does quantization change the results? At small scale, usually no — the top-K
structure survives compression. At large scale + tight quantization, results
can shift slightly.

In [40]:
query = TEST_QUESTIONS[0]
q_vec = embed_one(query)

print(f"Q: {query!r}\n")
for coll in [COLL_HNSW_DEFAULT, COLL_QUANT]:
    hits = qdrant.query_points(collection_name=coll, query=q_vec, limit=3).points
    label = "FULL PRECISION" if coll == COLL_HNSW_DEFAULT else "QUANTIZED (int8)"
    print(f"  {label}:")
    for h in hits:
        p = h.payload
        print(f"    {h.score:.3f}  {p['chunk_id']:8s} ({p['source']})")
    print()

Q: 'Where is the company located?'

  FULL PRECISION:
    0.458  Risk with CFDS 2.txt#6 ()
    0.457  General risk disclosure(1).txt#5 ()
    0.451  Risk with CFDS.txt#6 ()

  QUANTIZED (int8):
    0.458  Risk with CFDS 2.txt#6 ()
    0.457  General risk disclosure(1).txt#5 ()
    0.451  Risk with CFDS.txt#6 ()



**Discussion:**
- Same top-3? Same order? Same scores?
- The scores may differ slightly (compression is lossy) but the ranking
  usually holds.
- **Takeaway:** at large scale, quantization gives you 4× memory savings and
  faster queries with usually-negligible quality loss. It's a lever you
  reach for when you outgrow your memory budget.

---

## Cell 10 — Three similarity metrics on the same corpus

Now we shift from indices to metrics. Qdrant supports three:
- **Cosine** — measures angle. Range [-1, 1]. Scale-invariant.
- **Dot product** — measures angle + magnitude. Range depends on vectors.
- **L2 (Euclidean)** — straight-line distance. LOWER is more similar. Range [0, ∞).

We'll create three collections, one per metric, and query them all with the
same question.

In [41]:
COLL_COSINE = "wk07_day2_metric_cosine"
COLL_DOT    = "wk07_day2_metric_dot"
COLL_L2     = "wk07_day2_metric_l2"

recreate_collection(COLL_COSINE, distance=Distance.COSINE)
recreate_collection(COLL_DOT,    distance=Distance.DOT)
recreate_collection(COLL_L2,     distance=Distance.EUCLID)

upsert_corpus(COLL_COSINE)
upsert_corpus(COLL_DOT)
upsert_corpus(COLL_L2)

print("Created three collections, one per metric. All contain the same 10 animals.")

Uploaded    1 -  100 (100/1832)
Uploaded  101 -  200 (200/1832)
Uploaded  201 -  300 (300/1832)
Uploaded  301 -  400 (400/1832)
Uploaded  401 -  500 (500/1832)
Uploaded  501 -  600 (600/1832)
Uploaded  601 -  700 (700/1832)
Uploaded  701 -  800 (800/1832)
Uploaded  801 -  900 (900/1832)
Uploaded  901 - 1000 (1000/1832)
Uploaded 1001 - 1100 (1100/1832)
Uploaded 1101 - 1200 (1200/1832)
Uploaded 1201 - 1300 (1300/1832)
Uploaded 1301 - 1400 (1400/1832)
Uploaded 1401 - 1500 (1500/1832)
Uploaded 1501 - 1600 (1600/1832)
Uploaded 1601 - 1700 (1700/1832)
Uploaded 1701 - 1800 (1800/1832)
Uploaded 1801 - 1832 (1832/1832)
Uploaded    1 -  100 (100/1832)
Uploaded  101 -  200 (200/1832)
Uploaded  201 -  300 (300/1832)
Uploaded  301 -  400 (400/1832)
Uploaded  401 -  500 (500/1832)
Uploaded  501 -  600 (600/1832)
Uploaded  601 -  700 (700/1832)
Uploaded  701 -  800 (800/1832)
Uploaded  801 -  900 (900/1832)
Uploaded  901 - 1000 (1000/1832)
Uploaded 1001 - 1100 (1100/1832)
Uploaded 1101 - 1200 (1200/1

In [43]:
query = TEST_QUESTIONS[0]
q_vec = embed_one(query)

print(f"Q: {query!r}\n")
for coll, label in [(COLL_COSINE, "COSINE"),
                     (COLL_DOT,    "DOT PRODUCT"),
                     (COLL_L2,     "L2 (Euclidean, lower = closer)")]:
    hits = qdrant.query_points(collection_name=coll, query=q_vec, limit=3).points
    print(f"  {label}:")
    for h in hits:
        p = h.payload
        print(f"    {h.score:>7.3f}  {p['chunk_id']:8s} ({p['source']})")
    print()

Q: 'Where is the company located?'

  COSINE:
      0.458  Risk with CFDS 2.txt#6 ()
      0.457  General risk disclosure(1).txt#5 ()
      0.451  Risk with CFDS.txt#6 ()

  DOT PRODUCT:
      0.458  Risk with CFDS 2.txt#6 ()
      0.457  General risk disclosure(1).txt#5 ()
      0.451  Risk with CFDS.txt#6 ()

  L2 (Euclidean, lower = closer):
      1.041  Risk with CFDS 2.txt#6 ()
      1.042  General risk disclosure(1).txt#5 ()
      1.048  Risk with CFDS.txt#6 ()



**Discussion moment:**
- Did all three metrics return the same top-3 animals in the same order?
- The SCORES will differ (each metric produces a different number range).
  Only compare metrics via **rank order**, not raw scores.
- If the rankings agree, that's because OpenAI embeddings are **normalised**
  (unit-length). For unit vectors, cosine = dot product (up to sign), and L2
  is a monotonic function of cosine. So rankings match.
- **What if vectors AREN'T normalised?** Next cell.

---

## Cell 11 — Metric ranges compared

Same pair of vectors, all three metrics. See how the numbers differ.

In [44]:
# ------------------------------------------------------------
# Compare Cosine, Dot Product and L2 distance
# using our actual HFM corpus
# ------------------------------------------------------------

# Use first Golden Set question
query = TEST_QUESTIONS[0]

# Create query embedding
query_vec = embed_one(query)


# ------------------------------------------------------------
# Calculate similarity of the query against all corpus chunks
# ------------------------------------------------------------

scored_chunks = [
    (
        cosine(query_vec, doc["vector"]),
        doc
    )
    for doc in linear_corpus
]

# Highest cosine similarity first
scored_chunks.sort(
    key=lambda x: x[0],
    reverse=True
)


# Most relevant chunk
similar_doc = scored_chunks[0][1]

# Least similar chunk
different_doc = scored_chunks[-1][1]


# ------------------------------------------------------------
# Build comparison pairs
# ------------------------------------------------------------

pairs = [
    (
        "Query",
        "Most relevant chunk",
        query_vec,
        similar_doc["vector"],
        "semantically closer"
    ),

    (
        "Query",
        "Least relevant chunk",
        query_vec,
        different_doc["vector"],
        "semantically farther"
    )
]


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(f"Query: {query!r}\n")

print(
    f"  {'Pair':35s} "
    f"{'cosine':>10s} "
    f"{'dot':>10s} "
    f"{'l2':>10s}   "
    f"Interpretation"
)

print(
    f"  {'----':35s} "
    f"{'------':>10s} "
    f"{'---':>10s} "
    f"{'--':>10s}   "
    f"{'-' * 25}"
)


for a, b, va, vb, note in pairs:

    c = cosine(va, vb)
    d = dot(va, vb)
    l = l2(va, vb)

    pair_name = f"{a} vs {b}"

    print(
        f"  {pair_name:35s} "
        f"{c:>10.3f} "
        f"{d:>10.3f} "
        f"{l:>10.3f}   "
        f"{note}"
    )


# ------------------------------------------------------------
# Show which chunks were selected
# ------------------------------------------------------------

print()

print("Most relevant chunk:")
print(f"  ID:     {similar_doc['id']}")
print(f"  Source: {similar_doc['source']}")

print()

print("Least relevant chunk:")
print(f"  ID:     {different_doc['id']}")
print(f"  Source: {different_doc['source']}")

print()

print("Interpretation:")
print("  Cosine: higher = more similar")
print("  Dot:    higher = more similar")
print("  L2:     lower  = more similar")


# vec_cat   = next(d['vector'] for d in CORPUS_ANIMALS if d['id'] == 'cat')
# vec_dog   = next(d['vector'] for d in CORPUS_ANIMALS if d['id'] == 'dog')
# vec_bee   = next(d['vector'] for d in CORPUS_ANIMALS if d['id'] == 'bee')

# pairs = [("cat", "dog", vec_cat, vec_dog, "similar (both mammals)"),
#          ("cat", "bee", vec_cat, vec_bee, "different (mammal vs insect)")]

# print(f"  {'Pair':15s}  {'cosine':>8s}  {'dot':>8s}  {'l2':>8s}   Interpretation")
# print(f"  {'----':15s}  {'------':>8s}  {'---':>8s}  {'--':>8s}   {'-'*30}")
# for a, b, va, vb, note in pairs:
#     c = cosine(va, vb)
#     d = dot(va, vb)
#     l = l2(va, vb)
#     print(f"  {a:<5s} vs {b:6s}  {c:>8.3f}  {d:>8.3f}  {l:>8.3f}   {note}")

# print()
# print("Note: for OpenAI embeddings (unit-length), cosine ≈ dot product,")
# print("and L2 = sqrt(2 - 2*cosine).")
# print("Cosine higher = closer.  Dot higher = closer.  L2 lower = closer.")

Query: 'Where is the company located?'

  Pair                                    cosine        dot         l2   Interpretation
  ----                                    ------        ---         --   -------------------------
  Query vs Most relevant chunk             0.458      0.458      1.041   semantically closer
  Query vs Least relevant chunk            0.027      0.027      1.395   semantically farther

Most relevant chunk:
  ID:     Risk with CFDS 2.txt#6
  Source: 

Least relevant chunk:
  ID:     Account opening agreement 2(1).txt#40
  Source: 

Interpretation:
  Cosine: higher = more similar
  Dot:    higher = more similar
  L2:     lower  = more similar


---

## Cell 12 — Programme default: cosine. Why?

**Cosine is the safe default for text embeddings.** Reasons:

1. **Scale-invariant.** If you accidentally scale your vectors, cosine still
   works. Dot product doesn't.
2. **Bounded range [-1, 1].** Easy to reason about. L2 has no upper bound.
3. **Matches how embedding models are trained.** Most modern text embedding
   models are trained with cosine as the target metric.
4. **Universally supported.** Every vector DB has it.

**When you'd deviate:**
- **Dot product** — you know your embeddings are normalised AND you want a
  small speedup (one less division per comparison). Marginal at production scale.
- **L2** — you're working with non-text embeddings where magnitude matters
  (e.g. some image or audio embeddings). Rare in RAG.

**Programme rule:** stay on cosine unless you have a specific reason. Log the
reason in your ADR when you deviate.

---

## Cell 13 — The silent-bug pattern

Deck slide 25 warned about this. Let's build it live.

**Scenario:** you use a custom embedding pipeline that doesn't normalise its
output vectors. You configure Qdrant with `dot` distance because someone read
'dot is faster than cosine.' It looks like it works. Your KPIs pass.

But quietly, the rankings are wrong — because dot product favours HIGH-MAGNITUDE
vectors, and non-normalised embeddings vary in magnitude for no meaningful reason.

Let's simulate: take our animal vectors and scale a few of them up artificially.

In [45]:
# ------------------------------------------------------------
# Simulate non-normalised custom embeddings
# Scale a few actual corpus vectors by 3x
# ------------------------------------------------------------

scaled_docs = []

# Arbitrarily choose two chunks to scale
scaled_indices = {4, 9}

for idx, d in enumerate(linear_corpus):

    scale = 3.0 if idx in scaled_indices else 1.0

    vec = [
        v * scale
        for v in d["vector"]
    ]

    scaled_docs.append({
        **d,
        "vector": vec,
        "scale": scale
    })


# ------------------------------------------------------------
# Use our first Golden Set question
# ------------------------------------------------------------

query = TEST_QUESTIONS[0]

q_vec = embed_one(query)


# ------------------------------------------------------------
# Rank by cosine
#
# Cosine is scale-invariant, so multiplying a document
# vector by 3 should NOT improve its cosine similarity.
# ------------------------------------------------------------

cos_scored = [
    (
        cosine(q_vec, d["vector"]),
        d
    )
    for d in scaled_docs
]

cos_scored.sort(
    key=lambda p: p[0],
    reverse=True
)


# ------------------------------------------------------------
# Rank by dot product
#
# Dot product is affected by vector magnitude.
# Artificially scaled vectors may therefore move up
# in the ranking.
# ------------------------------------------------------------

dot_scored = [
    (
        dot(q_vec, d["vector"]),
        d
    )
    for d in scaled_docs
]

dot_scored.sort(
    key=lambda p: p[0],
    reverse=True
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(f"Q: {query!r}\n")


print("COSINE ranking (scale-invariant):")

for rank, (score, d) in enumerate(
    cos_scored[:5],
    1
):

    marker = (
        "  <-- scaled 3x"
        if d["scale"] > 1
        else ""
    )

    print(
        f"  [{rank}] "
        f"{score:>7.3f}  "
        f"{d['id']}  "
        f"source={d['source']}"
        f"{marker}"
    )


print(
    "\nDOT PRODUCT ranking "
    "(affected by vector magnitude):"
)

for rank, (score, d) in enumerate(
    dot_scored[:5],
    1
):

    marker = (
        "  <-- scaled 3x"
        if d["scale"] > 1
        else ""
    )

    print(
        f"  [{rank}] "
        f"{score:>7.3f}  "
        f"{d['id']}  "
        f"source={d['source']}"
        f"{marker}"
    )


# # Simulate non-normalised custom embeddings: scale some vectors up 3x for no reason
# scaled_docs = []
# for d in CORPUS_ANIMALS:
#     scale = 3.0 if d['id'] in ["salmon", "ant"] else 1.0   # arbitrarily scaled
#     vec = [v * scale for v in d['vector']]
#     scaled_docs.append({**d, "vector": vec, "scale": scale})

# # Query 'ocean animals' — the honest cosine ranking should put salmon and shark on top.
# query = "Which animals live in the ocean?"
# q_vec = embed_one(query)

# # Rank by cosine (correct — scale-invariant)
# cos_scored = [(cosine(q_vec, d['vector']), d) for d in scaled_docs]
# cos_scored.sort(key=lambda p: p[0], reverse=True)

# # Rank by dot product (WRONG when vectors aren't normalised — magnitude wins)
# dot_scored = [(dot(q_vec, d['vector']), d) for d in scaled_docs]
# dot_scored.sort(key=lambda p: p[0], reverse=True)

# print(f"Q: {query!r}\n")
# print("  COSINE ranking (correct):")
# for s, d in cos_scored[:5]:
#     marker = "  ← scaled 3x" if d['scale'] > 1 else ""
#     print(f"    {s:>7.3f}  {d['id']:8s} ({d['category']:7s}){marker}")

# print("\n  DOT PRODUCT ranking (buggy — magnitude wins over meaning):")
# for s, d in dot_scored[:5]:
#     marker = "  ← scaled 3x" if d['scale'] > 1 else ""
#     print(f"    {s:>7.3f}  {d['id']:8s} ({d['category']:7s}){marker}")

Q: 'Where is the company located?'

COSINE ranking (scale-invariant):
  [1]   0.458  Risk with CFDS 2.txt#6  source=
  [2]   0.457  General risk disclosure(1).txt#5  source=
  [3]   0.451  Risk with CFDS.txt#6  source=
  [4]   0.443  Complaint handling(1).txt#6  source=
  [5]   0.417  Client agreement(1).txt#454  source=

DOT PRODUCT ranking (affected by vector magnitude):
  [1]   0.679  Conflicts of interest policy (1).txt#9  source=  <-- scaled 3x
  [2]   0.543  Conflicts of interest policy (1).txt#4  source=  <-- scaled 3x
  [3]   0.459  Risk with CFDS 2.txt#6  source=
  [4]   0.457  General risk disclosure(1).txt#5  source=
  [5]   0.451  Risk with CFDS.txt#6  source=


**This is the silent bug.**
- Cosine returns salmon + shark at the top (correct — they're the actual ocean animals)
- Dot product returns salmon + ant at the top (wrong — ant scored high just because we scaled its vector)
- **In production this would look fine on some queries and quietly break others.**

**Prevention:**
1. **Always normalise your embedding vectors** if you're going to use dot product.
2. **Default to cosine** so this class of bug can't happen.
3. **Add a sanity check** that a vector's L2 norm is ~1.0 before upserting.

**Programme rule:** cosine. Unless you have a specific reason and have run
this exact test. Deck slide 26's takeaway.

---

## Cell 14 — Mini-RAG on top of Qdrant

Now assemble a full mini-RAG. Same shape as W6's `ask_rag()`, but Qdrant is
the storage. We'll use the cosine collection from Cell 10.

This mirrors exactly what you'll do in Track B for your capstone.

In [48]:
SYSTEM = (
    "You are a helpful assistant. Answer using ONLY the provided context. "
    "If the context does not contain the answer, say so plainly. Cite the "
    "animal id in square brackets after any fact you use."
)

def ask_qdrant_rag(question: str, collection: str, k: int = 3) -> dict:
    """Full mini-RAG: embed → Qdrant retrieve → prompt → generate."""
    # 1. Embed the query
    q_vec = embed_one(question)
    
    # 2. Retrieve top-K from Qdrant
    hits = qdrant.query_points(
        collection_name=collection,
        query=q_vec,
        limit=k,
    ).points
    
    # 3. Build the prompt with retrieved chunks
    context = "\n\n".join(
        f"[{h.payload.get('chunk_id', str(h.id))}]\n"
        f"Source: {h.payload.get('source', '')}\n"
        f"{h.payload.get('text', '')}"
        for h in hits
    )
    
    # 4. Generate
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content":
                f"Context:\n{context}\n\n---\n\nQuestion: {question}"},
        ],
    )
    
    return {
        "question": question,
        "answer":   resp.choices[0].message.content,
        "sources":  [h.payload["chunk_id"] for h in hits],
    }

# Test on one question
question = TEST_QUESTIONS[0]
result = ask_qdrant_rag(question, COLL_COSINE)
print(f"Q: {result['question']}\n")
print(f"A: {result['answer']}\n")
print(f"Sources: {result['sources']}")

Q: Where is the company located?

A: The company is located at Suite 305, Griffith Corporate Centre, P.O. Box 1510, Beachmont, Kingstown, Saint Vincent and the Grenadines. [Risk with CFDS 2.txt#6]

Sources: ['Risk with CFDS 2.txt#6', 'General risk disclosure(1).txt#5', 'Risk with CFDS.txt#6']


---

## Cell 15 — Run all 5 test questions

Now the full test set. Watch how well the mini-RAG handles each.

In [50]:
# for tq in TEST_QUESTIONS:
#     result = ask_qdrant_rag(tq["q"], COLL_COSINE, k=3)
#     print(f"── {tq['q']}")
#     print(f"   expected categories: {tq['expected_categories']}")
#     print(f"   sources retrieved:   {result['sources']}")
#     print(f"   answer: {result['answer']}")
#     print()


from pathlib import Path
import json

# ------------------------------------------------------------
# Load Golden Set
# ------------------------------------------------------------

golden_path = Path("./data/goldenSet.json")

with open(golden_path, "r", encoding="utf-8") as f:
    golden_set = json.load(f)

print(f"Loaded {len(golden_set)} Golden Set questions.")


# ------------------------------------------------------------
# Output file
# ------------------------------------------------------------

DATA_DIR = Path("./runs_output")
DATA_DIR.mkdir(parents=True, exist_ok=True)

output_file = DATA_DIR / "10_golden_set_rag_results.txt"


# ------------------------------------------------------------
# Run RAG against all Golden Set questions
# ------------------------------------------------------------

with open(output_file, "w", encoding="utf-8") as f:

    for item in golden_set:

        question_id = item["id"]
        question = item["question"]
        ideal_answer = item.get("ideal_answer", "")
        notes = item.get("notes", "")

        result = ask_qdrant_rag(
            question,
            COLL_COSINE,
            k=3
        )

        # Build output
        output = (
            f"Question ID: {question_id}\n"
            f"Question: {question}\n"
            f"Ideal Answer: {ideal_answer}\n"
            f"Sources Retrieved: {result['sources']}\n"
            f"RAG Answer: {result['answer']}\n"
            f"Notes: {notes}\n"
            f"{'-' * 100}\n\n"
        )

        # Print in notebook
        print(output)

        # Write to file
        f.write(output)


print(f"\nResults saved to: {output_file}")

Loaded 77 Golden Set questions.


Question ID: 1
Question: Where is the company located?
Ideal Answer: (a)	HFMarkets (SV) Ltd withregistered addressSuite 305,Griffith CorporateCentre,P.O. Box 1510, Beachmont Kingstown, St. Vincent and the Grenadines.
(b)	HF Markets (Europe) Ltd with registered address Spyrou Kyprianou 50, Irida 3 Tower 10th Floor, Larnaca 6057, Cyprus.
(c)	HFMarketsSA(PTY)LtdwithregisteredaddressKatherine&WestSuite18Secondfloor 114 West Street Sandton, Johannesburg 2031.
(d)	HFMarkets(Seychelles)LtdwithregisteredaddressRoom107,OrionComplex.POBox 1228, Victoria Mahe, Republic of Seychelles.
(e)	HF Markets Fintech Services Ltd with registered address Spyrou Kyprianou 50, Irida 3 Tower 10th Floor, Larnaca 6057, Cyprus.
(f)	HFMarketsLtdregulatedbytheFinancialServicesCommission(FSC)oftheRepublicof Mauritius, category 1 Global Business No. C110008214 License | Company Reg. No. 094286/GBL
(g)	HFMarkets(UK)LtdauthorisedandregulatedbytheFinancialConductAuthority(FCA) under firm reference number 801701.
Sources 

**Discussion:**
- Did every question retrieve at least one animal from the expected category?
- Which question was hardest?
- Did the LLM correctly cite `[animal_id]` for each fact it used?
- Look at Question 4 ("Which animals are kept as pets?") — did cat and dog
  come back? Did gecko?

---

## Cell 16 — Cleanup

Delete the collections we created today so your Qdrant cluster stays tidy.

(In your capstone Track B work you'll create ONE persistent collection —
`capstone_chunks`. These demo collections are transient.)

In [16]:
for coll in [COLL_HNSW_DEFAULT, COLL_HNSW_TUNED, COLL_QUANT,
             COLL_COSINE, COLL_DOT, COLL_L2]:
    try:
        qdrant.delete_collection(coll)
        print(f"  deleted {coll}")
    except Exception as e:
        print(f"  skip {coll}: {e}")

print("\nDay 1's animals collection (wk07_day1_animals) is preserved.")
print("Delete manually if you don't want it.")

  deleted wk07_day2_hnsw_default
  deleted wk07_day2_hnsw_tuned
  deleted wk07_day2_quantized
  deleted wk07_day2_metric_cosine
  deleted wk07_day2_metric_dot
  deleted wk07_day2_metric_l2

Day 1's animals collection (wk07_day1_animals) is preserved.
Delete manually if you don't want it.


---

## Cell 17 — Wrap: what you learned across two days

**Day 1 — Embedding space:**
1. Compared `3-small` vs `3-large`. 6.5× cost doesn't buy 6.5× quality.
2. Built intuition for semantic geometry — categories cluster.
3. Toured 4 vector DBs. Programme default: Qdrant.
4. Made your first Qdrant call.

**Day 2 — Qdrant tour:**
5. **Indices** exist because linear scan doesn't scale beyond ~10K docs.
6. **HNSW** is Qdrant's default. Two knobs: `m` (connectivity) and
   `ef_construct` (build depth). Both trade recall vs cost.
7. **Quantization** compresses vectors (int8 = 4× smaller). Pays off at scale.
8. **Cosine, dot, L2** — three distance metrics. Cosine is the safe default.
9. **The silent-bug pattern** — non-normalised vectors + dot product = wrong
   rankings that look fine. Stay on cosine.
10. **Mini-RAG on Qdrant** — same shape as W6's ask_rag, different storage.

**Track B (take-home): migrate YOUR capstone.**

Open `AI-RAG_W7_Application_Growth_Guide.md` and follow the steps. Same
pattern you just did with 10 animals, applied to your capstone's 100-300
real chunks. About 2 hours self-paced.

You will:
- Add `src/rag/qdrant_store.py` (~60 lines)
- Add `src/rag/qdrant_rag.py` (~80 lines) — parallel to W6's naive_rag
- Add `scripts/migrate_to_qdrant.py` — one-off migration
- Swap one function call in `src/api/main.py`
- Update `docs/adr/0001-capstone-framing.md` with vector-stack decisions
- Fill `docs/kpi/wk7-snapshot.md`